# Muscle Trainer: make the calm voice library
This makes every muscle name, nickname, what-it-does line and meditation phrase with the free **Kokoro** voice, and packs them into an `audio` folder for the website.

**How to run:** optional first: **Runtime → Change runtime type → T4 GPU** (free, much faster). Then **Runtime → Run all**. It takes about 15–25 minutes. When it finishes, your browser downloads `audio.zip`. Unzip it and upload the whole `audio` folder to the GitHub repository, next to `index.html`.

In [ ]:
# 1. Install Kokoro in its own Python 3.12 (a few minutes; Kokoro doesn't run on Colab's newer Python yet)
!pip install -q uv
!uv venv /content/kenv --python 3.12 -q
!uv pip install -q --python /content/kenv/bin/python 'kokoro>=0.9.4' soundfile numpy pip
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!/content/kenv/bin/python -c "import kokoro; print('installed')"

In [ ]:
# 2. Choose the voice and listen to a sample.
# Calm options: af_heart (warm, default), af_bella, af_nicole (soft, close-up), bf_emma (British), am_michael (male)
VOICE = 'af_heart'
SPEED = 0.9   # 1.0 = normal; lower is slower

open('/content/kgen.py', 'w').write("\n# Runs inside the Python 3.12 environment made in step 1 (Kokoro does not install on Colab's newer Python yet).\nimport sys, json, wave\nimport numpy as np\nfrom kokoro import KPipeline\nvoice, speed, mode = sys.argv[1], float(sys.argv[2]), sys.argv[3]\npipe = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M')\ndef tts(text):\n    parts = []\n    for r in pipe(text, voice=voice, speed=speed):\n        a = getattr(r, 'audio', None)\n        if a is None:\n            try: a = list(r)[2]\n            except Exception: a = None\n        if a is not None: parts.append(a.detach().cpu().numpy() if hasattr(a, 'detach') else np.asarray(a))\n    return np.concatenate(parts).astype(np.float32) if parts else np.zeros(2400, dtype=np.float32)\nif mode == 'sample':\n    a = tts(sys.argv[4])\n    with wave.open('/content/sample.wav', 'wb') as w:\n        w.setnchannels(1); w.setsampwidth(2); w.setframerate(24000)\n        w.writeframes((np.clip(a, -1, 1) * 32767).astype('<i2').tobytes())\n    print('sample ready')\nelse:\n    sys.path.insert(0, '/content')\n    from audio_build import build\n    data = json.load(open('/content/data.json', encoding='utf-8')); data['voice'] = voice\n    build(data, tts, '/content/audio')\n")
sample = "Breathe in, and on the out breath, we will recenter our minds on the... [gastrocnemius](/ɡˌæstɹɑknˈimiəs/)."
!/content/kenv/bin/python -u /content/kgen.py {VOICE} {SPEED} sample "{sample}"
from IPython.display import Audio, display
display(Audio('/content/sample.wav'))

In [ ]:
# 3. The script and the pronunciation table (made from the Muscle Trainer page; no need to edit)
import json
DATA = json.loads(r'''{"muscles": [{"id": "trap", "group": 4, "name": "Trapezius", "action": "Lifts, lowers and pulls back the shoulder blade", "nick": "Traps"}, {"id": "lat", "group": 4, "name": "Latissimus dorsi", "action": "Pulls the arm down and back", "nick": "Lats"}, {"id": "glutmax", "group": 10, "name": "Gluteus maximus", "action": "Extends the hip", "nick": "Glute max"}, {"id": "pec", "group": 3, "name": "Pectoralis major", "action": "Pulls the arm across the chest", "nick": "Pecs"}, {"id": "rect", "group": 5, "name": "Rectus abdominis", "action": "Bends the trunk forward", "nick": "Abs (the six-pack)"}, {"id": "obl", "group": 5, "name": "External oblique", "action": "Twists and bends the trunk", "nick": "Obliques"}, {"id": "serr", "group": 3, "name": "Serratus anterior", "action": "Holds the shoulder blade against the ribs", "nick": "Serratus (the boxer's muscle)"}, {"id": "erector", "group": 4, "name": "Erector spinae", "action": "Straightens the spine", "nick": "Erectors"}, {"id": "rhomb", "group": 4, "name": "Rhomboid major", "action": "Pulls the shoulder blades together", "nick": "Rhomboids"}, {"id": "glutmed", "group": 10, "name": "Gluteus medius", "action": "Moves the leg out to the side", "nick": "Glute med"}, {"id": "tfl", "group": 10, "name": "Tensor fasciae latae", "action": "Moves the leg out and steadies the hip", "nick": "TFL"}, {"id": "delt", "group": 2, "name": "Deltoid", "action": "Lifts the arm away from the body", "nick": "Delts"}, {"id": "supra", "group": 2, "name": "Supraspinatus", "action": "Starts lifting the arm", "nick": "Supra"}, {"id": "infra", "group": 2, "name": "Infraspinatus", "action": "Rotates the arm outward", "nick": "Infra"}, {"id": "tmin", "group": 2, "name": "Teres minor", "action": "Rotates the arm outward", "nick": null}, {"id": "tmaj", "group": 2, "name": "Teres major", "action": "Rotates the arm inward and pulls it back", "nick": null}, {"id": "scm", "group": 1, "name": "Sternocleidomastoid", "action": "Turns and tilts the head", "nick": "SCM"}, {"id": "plat", "group": 1, "name": "Platysma", "action": "Tightens the neck skin and pulls the mouth down", "nick": null}, {"id": "scal", "group": 1, "name": "Scalenes", "action": "Lifts the first ribs and tilts the neck", "nick": null}, {"id": "spl", "group": 1, "name": "Splenius capitis", "action": "Extends and turns the head", "nick": null}, {"id": "lev", "group": 1, "name": "Levator scapulae", "action": "Lifts the shoulder blade", "nick": "Levator scap"}, {"id": "front", "group": 0, "name": "Frontalis", "action": "Raises the eyebrows", "nick": null}, {"id": "orbi", "group": 0, "name": "Orbicularis oculi", "action": "Closes the eyelid", "nick": null}, {"id": "zyg", "group": 0, "name": "Zygomaticus major", "action": "Lifts the corner of the mouth (smiling)", "nick": "Smile muscle"}, {"id": "oris", "group": 0, "name": "Orbicularis oris", "action": "Closes and purses the lips", "nick": null}, {"id": "mass", "group": 0, "name": "Masseter", "action": "Closes the jaw to chew", "nick": null}, {"id": "temp", "group": 0, "name": "Temporalis", "action": "Closes the jaw", "nick": null}, {"id": "bic", "group": 6, "name": "Biceps brachii", "action": "Bends the elbow and turns the palm up", "nick": "Biceps (bis)"}, {"id": "bra", "group": 6, "name": "Brachialis", "action": "Bends the elbow", "nick": null}, {"id": "tric", "group": 6, "name": "Triceps brachii", "action": "Straightens the elbow", "nick": "Triceps (tris)"}, {"id": "brach", "group": 7, "name": "Brachioradialis", "action": "Bends the elbow", "nick": null}, {"id": "pron", "group": 7, "name": "Pronator teres", "action": "Turns the palm down", "nick": null}, {"id": "fcr", "group": 7, "name": "Flexor carpi radialis", "action": "Bends the wrist", "nick": null}, {"id": "extd", "group": 7, "name": "Extensor digitorum", "action": "Straightens the fingers", "nick": null}, {"id": "ecrl", "group": 7, "name": "Extensor carpi radialis longus", "action": "Bends the wrist backward", "nick": null}, {"id": "apb", "group": 8, "name": "Abductor pollicis brevis", "action": "Moves the thumb away from the palm", "nick": null}, {"id": "adm", "group": 8, "name": "Abductor digiti minimi", "action": "Moves the little finger away", "nick": null}, {"id": "adp", "group": 8, "name": "Adductor pollicis", "action": "Pulls the thumb toward the palm", "nick": null}, {"id": "dio", "group": 8, "name": "Dorsal interossei", "action": "Spreads the fingers", "nick": null}, {"id": "pect", "group": 9, "name": "Pectineus", "action": "Pulls the thigh inward and bends the hip", "nick": null}, {"id": "pirif", "group": 9, "name": "Piriformis", "action": "Rotates the thigh outward", "nick": null}, {"id": "rf", "group": 11, "name": "Rectus femoris", "action": "Straightens the knee and bends the hip", "nick": "Rec fem"}, {"id": "vl", "group": 11, "name": "Vastus lateralis", "action": "Straightens the knee", "nick": null}, {"id": "vm", "group": 11, "name": "Vastus medialis", "action": "Straightens the knee", "nick": "VMO (the teardrop)"}, {"id": "sart", "group": 11, "name": "Sartorius", "action": "Crosses the legs and bends the hip and knee", "nick": "Tailor's muscle"}, {"id": "add", "group": 11, "name": "Adductor longus", "action": "Pulls the thigh toward the middle", "nick": null}, {"id": "grac", "group": 11, "name": "Gracilis", "action": "Pulls the thigh inward and bends the knee", "nick": null}, {"id": "bf", "group": 11, "name": "Biceps femoris", "action": "Bends the knee and extends the hip", "nick": null}, {"id": "st", "group": 11, "name": "Semitendinosus", "action": "Bends the knee and extends the hip", "nick": null}, {"id": "sm", "group": 11, "name": "Semimembranosus", "action": "Bends the knee and extends the hip", "nick": null}, {"id": "ta", "group": 12, "name": "Tibialis anterior", "action": "Lifts the foot upward", "nick": "Tib ant (the shin muscle)"}, {"id": "fib", "group": 12, "name": "Fibularis longus", "action": "Turns the sole outward", "nick": null}, {"id": "edl", "group": 12, "name": "Extensor digitorum longus", "action": "Lifts the toes and foot", "nick": null}, {"id": "gas", "group": 12, "name": "Gastrocnemius", "action": "Points the foot down and bends the knee", "nick": "Gastroc"}, {"id": "sol", "group": 12, "name": "Soleus", "action": "Points the foot down", "nick": null}, {"id": "edb", "group": 13, "name": "Extensor digitorum brevis", "action": "Lifts the toes", "nick": null}, {"id": "ahal", "group": 13, "name": "Abductor hallucis", "action": "Moves the big toe inward", "nick": null}, {"id": "adm5", "group": 13, "name": "Abductor digiti minimi (foot)", "action": "Moves the little toe outward", "nick": null}, {"id": "fdb", "group": 13, "name": "Flexor digitorum brevis", "action": "Bends the toes", "nick": null}, {"id": "occip", "group": 0, "name": "Occipitalis", "action": "Pulls the scalp backward", "nick": null}, {"id": "corrug", "group": 0, "name": "Corrugator supercilii", "action": "Draws the eyebrows together and down (frowning)", "nick": null}, {"id": "proc", "group": 0, "name": "Procerus", "action": "Pulls the inner eyebrows down and wrinkles the bridge of the nose", "nick": null}, {"id": "nasal", "group": 0, "name": "Nasalis", "action": "Flares and narrows the nostrils", "nick": null}, {"id": "lls", "group": 0, "name": "Levator labii superioris", "action": "Lifts the upper lip", "nick": null}, {"id": "zygmin", "group": 0, "name": "Zygomaticus minor", "action": "Lifts the upper lip", "nick": null}, {"id": "dli", "group": 0, "name": "Depressor labii inferioris", "action": "Pulls the lower lip down", "nick": null}, {"id": "dao", "group": 0, "name": "Depressor anguli oris", "action": "Pulls the corner of the mouth down (frowning)", "nick": null}, {"id": "ment", "group": 0, "name": "Mentalis", "action": "Wrinkles the chin and pushes the lower lip out (pouting)", "nick": null}, {"id": "buc", "group": 0, "name": "Buccinator", "action": "Presses the cheek against the teeth (blowing, chewing)", "nick": null}, {"id": "ris", "group": 0, "name": "Risorius", "action": "Pulls the corner of the mouth sideways (grinning)", "nick": null}, {"id": "ssc", "group": 1, "name": "Semispinalis capitis", "action": "Extends the head and turns it slightly", "nick": null}, {"id": "subsc", "group": 2, "name": "Subscapularis", "action": "Rotates the arm inward", "nick": "Subscap"}, {"id": "pecmin", "group": 3, "name": "Pectoralis minor", "action": "Pulls the shoulder blade forward and down", "nick": "Pec minor"}, {"id": "subcl", "group": 3, "name": "Subclavius", "action": "Steadies the clavicle", "nick": null}, {"id": "eic", "group": 14, "name": "External intercostals", "action": "Lift the ribs to breathe in", "nick": null}, {"id": "iic", "group": 14, "name": "Internal intercostals", "action": "Pull the ribs down to breathe out hard", "nick": null}, {"id": "diaph", "group": 14, "name": "Diaphragm", "action": "The main breathing muscle: it flattens to draw air into the lungs", "nick": null}, {"id": "rhmin", "group": 4, "name": "Rhomboid minor", "action": "Pulls the shoulder blades together", "nick": "Rhomboids"}, {"id": "ql", "group": 4, "name": "Quadratus lumborum", "action": "Bends the trunk to the side and hitches up the hip", "nick": "QL"}, {"id": "iobl", "group": 5, "name": "Internal oblique", "action": "Twists and bends the trunk, and compresses the belly", "nick": "Obliques"}, {"id": "tabd", "group": 5, "name": "Transversus abdominis", "action": "Compresses the belly like a corset (core bracing)", "nick": "TA or TVA"}, {"id": "corbr", "group": 6, "name": "Coracobrachialis", "action": "Raises the arm forward and pulls it toward the body", "nick": null}, {"id": "pl", "group": 7, "name": "Palmaris longus", "action": "Tenses the palm and bends the wrist (missing in about 1 person in 7)", "nick": null}, {"id": "fcu", "group": 7, "name": "Flexor carpi ulnaris", "action": "Bends the wrist and tilts it toward the little finger", "nick": null}, {"id": "fds", "group": 7, "name": "Flexor digitorum superficialis", "action": "Bends the fingers at their middle joints", "nick": null}, {"id": "fdp", "group": 7, "name": "Flexor digitorum profundus", "action": "Bends the fingertips", "nick": null}, {"id": "fpl", "group": 7, "name": "Flexor pollicis longus", "action": "Bends the tip of the thumb", "nick": null}, {"id": "pq", "group": 7, "name": "Pronator quadratus", "action": "Turns the palm down", "nick": null}, {"id": "anc", "group": 7, "name": "Anconeus", "action": "Helps straighten the elbow and steadies it", "nick": null}, {"id": "ecrb", "group": 7, "name": "Extensor carpi radialis brevis", "action": "Bends the wrist backward", "nick": null}, {"id": "ecu", "group": 7, "name": "Extensor carpi ulnaris", "action": "Bends the wrist backward and tilts it toward the little finger", "nick": null}, {"id": "sup", "group": 7, "name": "Supinator", "action": "Turns the palm up", "nick": null}, {"id": "apl", "group": 7, "name": "Abductor pollicis longus", "action": "Moves the thumb outward, away from the hand", "nick": null}, {"id": "epl", "group": 7, "name": "Extensor pollicis longus", "action": "Straightens the tip of the thumb", "nick": null}, {"id": "glutmin", "group": 10, "name": "Gluteus minimus", "action": "Moves the leg out to the side and rotates the thigh inward", "nick": "Glute min"}, {"id": "vint", "group": 11, "name": "Vastus intermedius", "action": "Straightens the knee", "nick": null}, {"id": "addbrev", "group": 11, "name": "Adductor brevis", "action": "Pulls the thigh toward the middle", "nick": null}, {"id": "admag", "group": 11, "name": "Adductor magnus", "action": "Pulls the thigh toward the middle and helps extend the hip", "nick": null}, {"id": "plant", "group": 12, "name": "Plantaris", "action": "Weakly helps point the foot and bend the knee", "nick": null}, {"id": "popl", "group": 12, "name": "Popliteus", "action": "Unlocks the straightened knee by rotating it slightly", "nick": null}, {"id": "tibpost", "group": 12, "name": "Tibialis posterior", "action": "Points the foot, turns the sole inward and holds up the arch", "nick": "Tib post"}, {"id": "fdl", "group": 12, "name": "Flexor digitorum longus", "action": "Bends toes 2–5", "nick": null}, {"id": "fhl", "group": 12, "name": "Flexor hallucis longus", "action": "Bends the big toe (push-off when walking)", "nick": null}, {"id": "fibbr", "group": 12, "name": "Fibularis brevis", "action": "Turns the sole outward", "nick": null}, {"id": "ehl", "group": 12, "name": "Extensor hallucis longus", "action": "Lifts the big toe", "nick": null}, {"id": "epb", "group": 7, "name": "Extensor pollicis brevis", "action": "Straightens the thumb at its knuckle", "nick": null}, {"id": "ei", "group": 7, "name": "Extensor indicis", "action": "Straightens the index finger (pointing)", "nick": null}, {"id": "edm", "group": 7, "name": "Extensor digiti minimi", "action": "Straightens the little finger", "nick": null}, {"id": "fpb", "group": 8, "name": "Flexor pollicis brevis", "action": "Bends the thumb at its knuckle", "nick": null}, {"id": "op", "group": 8, "name": "Opponens pollicis", "action": "Brings the thumb across the palm to touch the fingertips", "nick": null}, {"id": "fdmb", "group": 8, "name": "Flexor digiti minimi brevis", "action": "Bends the little finger at its knuckle", "nick": null}, {"id": "odm", "group": 8, "name": "Opponens digiti minimi", "action": "Cups the palm by drawing the 5th metacarpal forward", "nick": null}, {"id": "pb", "group": 8, "name": "Palmaris brevis", "action": "Wrinkles the skin on the little-finger side of the palm and deepens the cup of the hand", "nick": null}, {"id": "lumb", "group": 8, "name": "Lumbricals", "action": "Bend the fingers at the knuckles while straightening the finger joints", "nick": null}, {"id": "pio", "group": 8, "name": "Palmar interossei", "action": "Pull fingers 2, 4 and 5 toward the middle finger", "nick": null}, {"id": "psoas", "group": 9, "name": "Psoas major", "action": "Bends the hip and the lower spine", "nick": "Psoas"}, {"id": "iliac", "group": 9, "name": "Iliacus", "action": "Bends the hip (with psoas major, as iliopsoas)", "nick": null}, {"id": "psmin", "group": 9, "name": "Psoas minor", "action": "Weakly bends the lower spine (missing in about 4 people in 10)", "nick": null}, {"id": "gems", "group": 10, "name": "Superior gemellus", "action": "Rotates the thigh outward", "nick": null}, {"id": "obi", "group": 10, "name": "Obturator internus", "action": "Rotates the thigh outward and steadies the hip", "nick": null}, {"id": "gemi", "group": 10, "name": "Inferior gemellus", "action": "Rotates the thigh outward", "nick": null}, {"id": "qf", "group": 10, "name": "Quadratus femoris", "action": "Rotates the thigh outward", "nick": null}, {"id": "obe", "group": 10, "name": "Obturator externus", "action": "Rotates the thigh outward", "nick": null}, {"id": "fibt", "group": 12, "name": "Fibularis tertius", "action": "Lifts the foot and turns the sole outward", "nick": null}, {"id": "ehb", "group": 13, "name": "Extensor hallucis brevis", "action": "Straightens the big toe", "nick": null}, {"id": "diof", "group": 13, "name": "Dorsal interossei (foot)", "action": "Spread the toes away from the 2nd toe", "nick": null}, {"id": "qp", "group": 13, "name": "Quadratus plantae", "action": "Helps flexor digitorum longus bend the toes, pulling its line straight", "nick": null}, {"id": "lumbf", "group": 13, "name": "Lumbricals (foot)", "action": "Bend the toes at their bases while straightening the toe joints", "nick": null}, {"id": "fhb", "group": 13, "name": "Flexor hallucis brevis", "action": "Bends the big toe at its base", "nick": null}, {"id": "addh", "group": 13, "name": "Adductor hallucis", "action": "Pulls the big toe toward the 2nd toe and supports the arch", "nick": null}, {"id": "fdmbf", "group": 13, "name": "Flexor digiti minimi brevis (foot)", "action": "Bends the little toe at its base", "nick": null}, {"id": "plio", "group": 13, "name": "Plantar interossei", "action": "Pull toes 3–5 toward the 2nd toe", "nick": null}, {"id": "lcap", "group": 1, "name": "Longus capitis", "action": "Bends the head forward", "nick": null}, {"id": "lcol", "group": 1, "name": "Longus colli", "action": "Bends the neck forward", "nick": null}, {"id": "dig", "group": 1, "name": "Digastric", "action": "Opens the jaw and lifts the hyoid bone (two bellies joined by a tendon)", "nick": null}, {"id": "styh", "group": 1, "name": "Stylohyoid", "action": "Pulls the hyoid bone up and back when swallowing", "nick": null}, {"id": "myh", "group": 1, "name": "Mylohyoid", "action": "Forms the floor of the mouth; lifts the floor and the tongue when swallowing", "nick": null}, {"id": "genh", "group": 1, "name": "Geniohyoid", "action": "Pulls the hyoid bone forward and helps open the jaw", "nick": null}, {"id": "sth", "group": 1, "name": "Sternohyoid", "action": "Pulls the hyoid bone down after swallowing", "nick": null}, {"id": "stt", "group": 1, "name": "Sternothyroid", "action": "Pulls the voice box (larynx) down", "nick": null}, {"id": "thh", "group": 1, "name": "Thyrohyoid", "action": "Pulls the hyoid bone down, or lifts the voice box", "nick": null}, {"id": "omoh", "group": 1, "name": "Omohyoid", "action": "Pulls the hyoid bone down (two bellies joined by a tendon)", "nick": null}, {"id": "scapost", "group": 1, "name": "Posterior scalene", "action": "Lifts the 2nd rib and bends the neck to the side", "nick": null}, {"id": "scamid", "group": 1, "name": "Middle scalene", "action": "Lifts the 1st rib and bends the neck to the side", "nick": null}, {"id": "scaant", "group": 1, "name": "Anterior scalene", "action": "Lifts the 1st rib and bends the neck to the side", "nick": null}, {"id": "rcpma", "group": 1, "name": "Rectus capitis posterior major", "action": "Tilts the head back and turns it to the same side", "nick": null}, {"id": "rcpmi", "group": 1, "name": "Rectus capitis posterior minor", "action": "Tilts the head back", "nick": null}, {"id": "ocs", "group": 1, "name": "Obliquus capitis superior", "action": "Tilts the head back and to the side", "nick": null}, {"id": "oci", "group": 1, "name": "Obliquus capitis inferior", "action": "Turns the head to the same side", "nick": null}, {"id": "splc", "group": 4, "name": "Splenius cervicis", "action": "Extends the neck and turns it to the same side", "nick": null}, {"id": "sps", "group": 4, "name": "Serratus posterior superior", "action": "Lifts the upper ribs (breathing in)", "nick": null}, {"id": "spi", "group": 4, "name": "Serratus posterior inferior", "action": "Pulls the lower ribs down and back", "nick": null}, {"id": "ilco", "group": 4, "name": "Iliocostalis", "action": "Extends and bends the spine to the side (outer column of the erector spinae)", "nick": null}, {"id": "long", "group": 4, "name": "Longissimus", "action": "Extends the spine and the head (middle column of the erector spinae)", "nick": null}, {"id": "spinl", "group": 4, "name": "Spinalis", "action": "Extends the spine (inner column of the erector spinae)", "nick": null}, {"id": "lao", "group": 0, "name": "Levator anguli oris", "action": "Lifts the corner of the mouth", "nick": null}, {"id": "llsan", "group": 0, "name": "Levator labii superioris alaeque nasi", "action": "Lifts the upper lip and flares the nostril", "nick": null}, {"id": "dsn", "group": 0, "name": "Depressor septi nasi", "action": "Pulls the tip of the nose down", "nick": null}, {"id": "aura", "group": 0, "name": "Auricularis anterior", "action": "Pulls the ear forward (slightly)", "nick": null}, {"id": "aurs", "group": 0, "name": "Auricularis superior", "action": "Lifts the ear (slightly)", "nick": null}, {"id": "aurp", "group": 0, "name": "Auricularis posterior", "action": "Pulls the ear back (slightly)", "nick": null}, {"id": "ptm", "group": 0, "name": "Medial pterygoid", "action": "Closes the jaw and moves it side to side", "nick": null}, {"id": "ptl", "group": 0, "name": "Lateral pterygoid", "action": "Pulls the jaw forward and opens it; moves it side to side", "nick": null}, {"id": "pyr", "group": 5, "name": "Pyramidalis", "action": "Tenses the linea alba (missing in about 1 person in 5)", "nick": null}, {"id": "ttx", "group": 14, "name": "Transversus thoracis", "action": "Pulls the rib cartilages down (breathing out hard)", "nick": null}, {"id": "lps", "group": 15, "name": "Levator palpebrae superioris", "action": "Lifts the upper eyelid", "nick": null}, {"id": "rsup", "group": 15, "name": "Superior rectus", "action": "Turns the eye upward (and slightly inward)", "nick": null}, {"id": "rmed", "group": 15, "name": "Medial rectus", "action": "Turns the eye inward, toward the nose", "nick": null}, {"id": "rlat", "group": 15, "name": "Lateral rectus", "action": "Turns the eye outward", "nick": null}, {"id": "osup", "group": 15, "name": "Superior oblique", "action": "Turns the eye down and out, and rotates it inward (its tendon loops through the trochlea)", "nick": null}, {"id": "rinf", "group": 15, "name": "Inferior rectus", "action": "Turns the eye downward (and slightly inward)", "nick": null}, {"id": "oinf", "group": 15, "name": "Inferior oblique", "action": "Turns the eye up and out, and rotates it outward", "nick": null}, {"id": "hmaj", "group": 16, "name": "Helicis major", "action": "Tenses the front of the ear's rim (very slightly)", "nick": null}, {"id": "hmin", "group": 16, "name": "Helicis minor", "action": "Tenses the crus of the helix (very slightly)", "nick": null}, {"id": "trag", "group": 16, "name": "Tragicus", "action": "Pulls the tragus over the ear canal (very slightly)", "nick": null}, {"id": "atrag", "group": 16, "name": "Antitragicus", "action": "Bends the antitragus outward (very slightly)", "nick": null}, {"id": "taur", "group": 16, "name": "Transverse muscle of the auricle", "action": "Pulls the ear's rim toward the concha (very slightly)", "nick": null}, {"id": "oaur", "group": 16, "name": "Oblique muscle of the auricle", "action": "Pulls parts of the ear cartilage together (very slightly)", "nick": null}, {"id": "tty", "group": 16, "name": "Tensor tympani", "action": "Tightens the eardrum to damp loud sounds", "nick": null}, {"id": "stap", "group": 16, "name": "Stapedius", "action": "Steadies the stapes to damp loud sounds (the smallest skeletal muscle)", "nick": null}, {"id": "hg", "group": 17, "name": "Hyoglossus", "action": "Pulls the tongue down and back", "nick": null}, {"id": "sg", "group": 17, "name": "Styloglossus", "action": "Pulls the tongue up and back", "nick": null}, {"id": "pg", "group": 17, "name": "Palatoglossus", "action": "Lifts the back of the tongue and narrows the throat opening", "nick": null}, {"id": "uvu", "group": 17, "name": "Musculus uvulae", "action": "Shortens and lifts the uvula", "nick": null}, {"id": "gg", "group": 17, "name": "Genioglossus", "action": "Pushes the tongue forward (sticks it out) and holds it down", "nick": null}, {"id": "tvp", "group": 17, "name": "Tensor veli palatini", "action": "Tightens the soft palate and opens the auditory tube (when yawning or swallowing)", "nick": null}, {"id": "lvp", "group": 17, "name": "Levator veli palatini", "action": "Lifts the soft palate when swallowing", "nick": null}, {"id": "slong", "group": 17, "name": "Superior longitudinal muscle of the tongue", "action": "Shortens the tongue and curls its tip upward", "nick": null}, {"id": "ilong", "group": 17, "name": "Inferior longitudinal muscle of the tongue", "action": "Shortens the tongue and curls its tip downward", "nick": null}, {"id": "ttr", "group": 17, "name": "Transverse muscle of the tongue", "action": "Narrows and lengthens the tongue", "nick": null}, {"id": "tvt", "group": 17, "name": "Vertical muscle of the tongue", "action": "Flattens and widens the tongue", "nick": null}, {"id": "ipc", "group": 18, "name": "Inferior pharyngeal constrictor", "action": "Squeezes food down the lower pharynx when swallowing", "nick": null}, {"id": "pca", "group": 18, "name": "Posterior cricoarytenoid", "action": "Opens the vocal cords (the only muscle that does) so you can breathe", "nick": null}, {"id": "oblary", "group": 18, "name": "Oblique arytenoid", "action": "Closes the entrance to the larynx when swallowing", "nick": null}, {"id": "mpc", "group": 18, "name": "Middle pharyngeal constrictor", "action": "Squeezes food down the middle pharynx when swallowing", "nick": null}, {"id": "tranary", "group": 18, "name": "Transverse arytenoid", "action": "Pulls the arytenoid cartilages together, closing the back of the vocal cords", "nick": null}, {"id": "spc", "group": 18, "name": "Superior pharyngeal constrictor", "action": "Squeezes food down the upper pharynx when swallowing", "nick": null}, {"id": "styph", "group": 18, "name": "Stylopharyngeus", "action": "Lifts and widens the pharynx when swallowing", "nick": null}, {"id": "cthy", "group": 18, "name": "Cricothyroid", "action": "Tilts the thyroid cartilage to stretch the vocal cords (raises the pitch of the voice)", "nick": null}, {"id": "lca", "group": 18, "name": "Lateral cricoarytenoid", "action": "Brings the vocal cords together", "nick": null}, {"id": "salph", "group": 18, "name": "Salpingopharyngeus", "action": "Lifts the pharynx and helps open the auditory tube when swallowing", "nick": null}, {"id": "palph", "group": 18, "name": "Palatopharyngeus", "action": "Lifts the pharynx and larynx and narrows the throat opening when swallowing", "nick": null}, {"id": "tharyt", "group": 18, "name": "Thyroarytenoid", "action": "Shortens and relaxes the vocal cords (lowers the pitch)", "nick": null}, {"id": "voc", "group": 18, "name": "Vocalis", "action": "Fine-tunes the tension of the vocal cords", "nick": null}, {"id": "eas", "group": 19, "name": "External anal sphincter", "action": "Keeps the anus closed", "nick": null}, {"id": "bsp", "group": 19, "name": "Bulbospongiosus", "action": "Tightens around the vaginal opening or the urethra (empties the urethra)", "nick": null}, {"id": "isc", "group": 19, "name": "Ischiocavernosus", "action": "Helps keep the clitoris or penis erect", "nick": null}, {"id": "stp", "group": 19, "name": "Superficial transverse perineal", "action": "Steadies the perineal body", "nick": null}, {"id": "dtp", "group": 19, "name": "Deep transverse perineal", "action": "Steadies the perineal body and supports the pelvic floor", "nick": null}, {"id": "eus", "group": 19, "name": "External urethral sphincter", "action": "Keeps the urethra closed (holds in urine)", "nick": null}, {"id": "pbr", "group": 19, "name": "Puborectalis", "action": "Pulls the rectum forward into a bend, helping hold in stool", "nick": null}, {"id": "pcg", "group": 19, "name": "Pubococcygeus", "action": "Supports the pelvic organs (part of levator ani)", "nick": "PC muscle"}, {"id": "icg", "group": 19, "name": "Iliococcygeus", "action": "Supports the pelvic organs (part of levator ani)", "nick": null}, {"id": "ccg", "group": 19, "name": "Ischiococcygeus (coccygeus)", "action": "Supports the pelvic floor and the coccyx", "nick": null}, {"id": "tpar", "group": 0, "name": "Temporoparietalis", "action": "Tightens the scalp at the side of the head", "nick": null}, {"id": "dsup", "group": 0, "name": "Depressor supercilii", "action": "Pulls the inner eyebrow down", "nick": null}, {"id": "rca", "group": 1, "name": "Rectus capitis anterior", "action": "Bends the head forward", "nick": null}, {"id": "rcl", "group": 1, "name": "Rectus capitis lateralis", "action": "Bends the head to the side", "nick": null}, {"id": "ssce", "group": 4, "name": "Semispinalis cervicis", "action": "Extends the neck and turns it to the other side", "nick": null}, {"id": "sst", "group": 4, "name": "Semispinalis thoracis", "action": "Extends the upper back and turns it to the other side", "nick": null}, {"id": "rot", "group": 4, "name": "Rotatores", "action": "Steady the vertebrae and help twist the spine", "nick": null}, {"id": "mult", "group": 4, "name": "Multifidus", "action": "Steadies each vertebra and extends the spine", "nick": null}, {"id": "isp", "group": 4, "name": "Interspinales", "action": "Extend the spine (pairs between neighbouring spinous processes)", "nick": null}, {"id": "itr", "group": 4, "name": "Intertransversarii", "action": "Bend the spine to the side (between neighbouring transverse processes)", "nick": null}, {"id": "lcos", "group": 4, "name": "Levatores costarum", "action": "Lift the ribs", "nick": null}, {"id": "subco", "group": 14, "name": "Subcostales", "action": "Pull the lower ribs down (breathing out)", "nick": null}, {"id": "iicm", "group": 14, "name": "Innermost intercostals", "action": "Help the internal intercostals pull the ribs down (breathing out)", "nick": null}, {"id": "crem", "group": 5, "name": "Cremaster", "action": "Raises the testis (in males; only a few fibres in females)", "nick": null}, {"id": "artg", "group": 11, "name": "Articularis genus", "action": "Pulls the knee joint capsule up when the knee straightens", "nick": null}], "words": {"abdominis": ["ab dom in iss", "æbdˈɑmənɪs"], "abductor": ["abductor", "æbdˈʌktəɹ"], "adductor": ["adductor", "ədˈʌktəɹ"], "alaeque": ["uh lee kwee", "əlˈikwi"], "anal": ["anal", "ˈAnᵊl"], "anconeus": ["an cone ee us", "æŋkˈOniəs"], "anguli": ["ang gew lie", "ˈæŋɡjəlI"], "anterior": ["anterior", "ænˈtɪɹiəɹ"], "antitragicus": ["anti traj ick us", "ˌæntitɹˈæʤɪkəs"], "articularis": ["ar tick you lair iss", "ɑɹtˌɪkjəlˈɛɹɪs"], "arytenoid": ["air uh tee noid", "ˌɛɹətˈinYd"], "auricle": ["auricle", "ˈɔɹɪkᵊl"], "auricularis": ["or ick you lair iss", "ɔɹˌɪkjəlˈɛɹɪs"], "biceps": ["biceps", "bˈIsɛps"], "brachialis": ["bray key al iss", "bɹˌAkiˈælɪs"], "brachii": ["bray key eye", "bɹˈAkiI"], "brachioradialis": ["bray key oh ray dee al iss", "bɹˌAkiOɹˌAdiˈælɪs"], "brevis": ["brev iss", "bɹˈɛvɪs"], "buccinator": ["buck sin ay ter", "bˈʌksɪnˌAtəɹ"], "bulbospongiosus": ["bulb oh spun jee oh suss", "bˌʌlbOspˌʌnʤiˈOsəs"], "capitis": ["cap it iss", "kˈæpɪtɪs"], "carpi": ["car pie", "kˈɑɹpI"], "cervicis": ["ser viss iss", "sˈɜɹvɪsɪs"], "coccygeus": ["cock sidge ee us", "kɑksˈɪʤiəs"], "colli": ["coll eye", "kˈɑlI"], "constrictor": ["constrictor", "kənstɹˈɪktəɹ"], "coracobrachialis": ["core uh co bray key al iss", "kˌɔɹəkObɹˌAkiˈælɪs"], "corrugator": ["core uh gay ter", "kˈɔɹəɡˌAtəɹ"], "costarum": ["coss tair um", "kɑstˈɛɹəm"], "cremaster": ["cree master", "kɹimˈæstəɹ"], "cricoarytenoid": ["cry co air uh tee noid", "kɹˌIkOˌɛɹətˈinYd"], "cricothyroid": ["cry co thy roid", "kɹˌIkOθˈIɹYd"], "deltoid": ["deltoid", "dˈɛltYd"], "depressor": ["depressor", "dəpɹˈɛsəɹ"], "diaphragm": ["diaphragm", "dˈIəfɹˌæm"], "digastric": ["die gas trick", "dIɡˈæstɹɪk"], "digiti": ["didge it eye", "dˈɪʤɪtI"], "digitorum": ["didge it tore um", "dˌɪʤɪtˈɔɹəm"], "dorsal": ["dorsal", "dˈɔɹsᵊl"], "dorsi": ["door sigh", "dˈɔɹsI"], "erector": ["erector", "ɪɹˈɛktəɹ"], "extensor": ["extensor", "ɪkstˈɛnsəɹ"], "external": ["external", "ɪkstˈɜɹnᵊl"], "externus": ["ex turn us", "ɪkstˈɜɹnəs"], "fasciae": ["fash ee ee", "fˈæʃii"], "femoris": ["fem or iss", "fˈɛməɹɪs"], "fibularis": ["fib you lair iss", "fˌɪbjəlˈɛɹɪs"], "flexor": ["flexor", "flˈɛksəɹ"], "frontalis": ["fron tal iss", "fɹɑntˈælɪs"], "gastrocnemius": ["gas trock knee me us", "ɡˌæstɹɑknˈimiəs"], "gemellus": ["juh mell us", "ʤəmˈɛləs"], "genioglossus": ["jee nee oh gloss us", "ʤˌiniOɡlˈɑsəs"], "geniohyoid": ["jee nee oh high oid", "ʤˌiniOhˈIYd"], "genus": ["jee nuss", "ʤˈinəs"], "gluteus": ["glue tee us", "ɡlˈutiəs"], "gracilis": ["grass ill iss", "ɡɹˈæsɪlɪs"], "hallucis": ["hal uh siss", "hˈæləsɪs"], "helicis": ["hell iss iss", "hˈɛlɪsɪs"], "hyoglossus": ["high oh gloss us", "hˌIOɡlˈɑsəs"], "iliacus": ["ill eye uh cuss", "ɪlˈIəkəs"], "iliococcygeus": ["ill ee oh cock sidge ee us", "ˌɪliOkɑksˈɪʤiəs"], "iliocostalis": ["ill ee oh coss tal iss", "ˌɪliOkɑstˈælɪs"], "indicis": ["in dis iss", "ˈɪndɪsɪs"], "inferior": ["inferior", "ɪnfˈɪɹiəɹ"], "inferioris": ["in fear ee or iss", "ɪnfˌɪɹiˈɔɹɪs"], "infraspinatus": ["in fra spy nay tus", "ˌɪnfɹəspInˈAtəs"], "intercostals": ["inter coss tulls", "ˌɪntəɹkˈɑstᵊlz"], "intermedius": ["inter me dee us", "ˌɪntəɹmˈidiəs"], "internal": ["internal", "ɪntˈɜɹnᵊl"], "internus": ["in turn us", "ɪntˈɜɹnəs"], "interossei": ["inter oss ee eye", "ˌɪntəɹˈɑsiI"], "interspinales": ["inter spy nay lees", "ˌɪntəɹspInˈAliz"], "intertransversarii": ["inter trans ver sair ee eye", "ˌɪntəɹtɹˌænzvəɹsˈɛɹiI"], "ischiocavernosus": ["iss key oh cav er no suss", "ˌɪskiOkˌævəɹnˈOsəs"], "ischiococcygeus": ["iss key oh cock sidge ee us", "ˌɪskiOkɑksˈɪʤiəs"], "labii": ["lay bee eye", "lˈAbiI"], "latae": ["lay tee", "lˈAti"], "lateral": ["lateral", "lˈæɾəɹəl"], "lateralis": ["lat er al iss", "lˌæɾəɹˈælɪs"], "latissimus": ["luh tiss im us", "lətˈɪsəməs"], "levator": ["luh vay ter", "ləvˈAtəɹ"], "levatores": ["lev uh tore ees", "lˌɛvətˈɔɹiz"], "longissimus": ["lon jiss im us", "lɑnʤˈɪsəməs"], "longus": ["long gus", "lˈɔŋɡəs"], "lumborum": ["lum bore um", "lʌmbˈɔɹəm"], "lumbricals": ["lum brick ulls", "lˈʌmbɹɪkᵊlz"], "magnus": ["mag nus", "mˈæɡnəs"], "masseter": ["muh see ter", "məsˈitəɹ"], "maximus": ["maximus", "mˈæksəməs"], "medial": ["medial", "mˈidiəl"], "medialis": ["me dee al iss", "mˌidiˈælɪs"], "medius": ["me dee us", "mˈidiəs"], "mentalis": ["men tal iss", "mɛntˈælɪs"], "minimi": ["min im eye", "mˈɪnɪmI"], "minimus": ["min im us", "mˈɪnɪməs"], "multifidus": ["mull tiff id us", "mʌltˈɪfɪdəs"], "musculus": ["muss queue lus", "mˈʌskjələs"], "mylohyoid": ["my low high oid", "mˌIlOhˈIYd"], "nasalis": ["nay zal iss", "nAzˈælɪs"], "nasi": ["nay sigh", "nˈAzI"], "oblique": ["oblique", "Oblˈik"], "obliquus": ["oh blee kwus", "Oblˈikwəs"], "obturator": ["ob ter ray ter", "ˈɑbtəɹˌAtəɹ"], "occipitalis": ["ox sip it al iss", "ɑksˌɪpɪtˈælɪs"], "oculi": ["ock you lie", "ˈɑkjəlI"], "omohyoid": ["oh mo high oid", "ˌOmOhˈIYd"], "opponens": ["uh pone ens", "əpˈOnənz"], "orbicularis": ["or bick you lair iss", "ɔɹbˌɪkjəlˈɛɹɪs"], "oris": ["or iss", "ˈɔɹɪs"], "palatini": ["pal uh tie nigh", "pˌælətˈInI"], "palatoglossus": ["pal uh toe gloss us", "pˌælətOɡlˈɑsəs"], "palatopharyngeus": ["pal uh toe fuh rin jee us", "pˌælətOfəɹˈɪnʤiəs"], "palmar": ["pal mer", "pˈɑlməɹ"], "palmaris": ["pal mair iss", "pælmˈɛɹɪs"], "palpebrae": ["pal puh bree", "pˈælpəbɹi"], "pectineus": ["peck tin ee us", "pɛktˈɪniəs"], "pectoralis": ["peck tor al iss", "pˌɛktəɹˈælɪs"], "perineal": ["pair in ee ul", "pˌɛɹənˈiəl"], "pharyngeal": ["fuh rin jee ul", "fəɹˈɪnʤiəl"], "piriformis": ["pier if or miss", "pˌɪɹɪfˈɔɹmɪs"], "plantae": ["plan tee", "plˈænti"], "plantar": ["plan ter", "plˈæntəɹ"], "plantaris": ["plan tair iss", "plæntˈɛɹɪs"], "platysma": ["pluh tiz muh", "plətˈɪzmə"], "pollicis": ["poll iss iss", "pˈɑlɪsɪs"], "popliteus": ["pop lit ee us", "pɑplˈɪtiəs"], "posterior": ["posterior", "pɑstˈɪɹiəɹ"], "procerus": ["pro seer us", "pɹOsˈɪɹəs"], "profundus": ["pro fun dus", "pɹOfˈʌndəs"], "pronator": ["pro nay ter", "pɹˈOnˌAtəɹ"], "psoas": ["so us", "sˈOəs"], "pterygoid": ["terry goid", "tˈɛɹəɡYd"], "pubococcygeus": ["pew bo cock sidge ee us", "pjˌubOkɑksˈɪʤiəs"], "puborectalis": ["pew bo reck tal iss", "pjˌubOɹɛktˈælɪs"], "pyramidalis": ["pih ram id al iss", "pɪɹˌæmɪdˈælɪs"], "quadratus": ["quad ray tus", "kwɑdɹˈAtəs"], "radialis": ["ray dee al iss", "ɹˌAdiˈælɪs"], "rectus": ["wreck tus", "ɹˈɛktəs"], "rhomboid": ["rhomboid", "ɹˈɑmbYd"], "risorius": ["riss or ee us", "ɹɪsˈɔɹiəs"], "rotatores": ["row tuh tore ees", "ɹˌOtətˈɔɹiz"], "salpingopharyngeus": ["sal ping go fuh rin jee us", "sˌælpɪŋɡOfəɹˈɪnʤiəs"], "sartorius": ["sar tore ee us", "sɑɹtˈɔɹiəs"], "scalene": ["scay leen", "skˈAlin"], "scalenes": ["scay leens", "skˈAlinz"], "scapulae": ["scap you lee", "skˈæpjəli"], "semimembranosus": ["semmy mem bruh no suss", "sˌɛmimˌɛmbɹənˈOsəs"], "semispinalis": ["semmy spy nal iss", "sˌɛmispInˈælɪs"], "semitendinosus": ["semmy ten din oh suss", "sˌɛmitˌɛndɪnˈOsəs"], "septi": ["sep tie", "sˈɛptI"], "serratus": ["sir ay tus", "sɛɹˈAtəs"], "soleus": ["so lee us", "sˈOliəs"], "sphincter": ["sphincter", "sfˈɪŋktəɹ"], "spinae": ["spy nee", "spˈIni"], "spinalis": ["spy nal iss", "spInˈælɪs"], "splenius": ["splee nee us", "splˈiniəs"], "stapedius": ["stay pee dee us", "stApˈidiəs"], "sternocleidomastoid": ["stern oh cly doe mass toid", "stˌɜɹnOklˌIdOmˈæstYd"], "sternohyoid": ["stern oh high oid", "stˌɜɹnOhˈIYd"], "sternothyroid": ["stern oh thy roid", "stˌɜɹnOθˈIɹYd"], "styloglossus": ["sty low gloss us", "stˌIlOɡlˈɑsəs"], "stylohyoid": ["sty low high oid", "stˌIlOhˈIYd"], "stylopharyngeus": ["sty low fuh rin jee us", "stˌIlOfəɹˈɪnʤiəs"], "subclavius": ["sub clay vee us", "sʌbklˈAviəs"], "subcostales": ["sub coss tay lees", "sˌʌbkɑstˈAliz"], "subscapularis": ["sub scap you lair iss", "sˌʌbskæpjəlˈɛɹɪs"], "supercilii": ["super sill ee eye", "sˌupəɹsˈɪliI"], "superficialis": ["super fish ee al iss", "sˌupəɹfˌɪʃiˈælɪs"], "superior": ["superior", "supˈɪɹiəɹ"], "superioris": ["sue peer ee or iss", "supˌɪɹiˈɔɹɪs"], "supinator": ["sue pin ay ter", "sˈupɪnˌAtəɹ"], "supraspinatus": ["sue pra spy nay tus", "sˌupɹəspInˈAtəs"], "temporalis": ["tem por al iss", "tˌɛmpəɹˈælɪs"], "temporoparietalis": ["tem por oh puh rye uh tal iss", "tˌɛmpəɹOpəɹˌIətˈælɪs"], "tensor": ["tensor", "tˈɛnsəɹ"], "teres": ["tear ease", "tˈɪɹiz"], "tertius": ["ter shee us", "tˈɜɹʃiəs"], "thoracis": ["thor ay sis", "θɔɹˈAsɪs"], "thyroarytenoid": ["thy row air uh tee noid", "θˌIɹOˌɛɹətˈinYd"], "thyrohyoid": ["thy row high oid", "θˌIɹOhˈIYd"], "tibialis": ["tib ee al iss", "tˌɪbiˈælɪs"], "tragicus": ["traj ick us", "tɹˈæʤɪkəs"], "transverse": ["transverse", "tɹænzvˈɜɹs"], "transversus": ["trans ver suss", "tɹænzvˈɜɹsəs"], "trapezius": ["truh pee zee us", "tɹəpˈiziəs"], "triceps": ["triceps", "tɹˈIsɛps"], "tympani": ["tim puh nigh", "tˈɪmpənI"], "ulnaris": ["ull nair iss", "ʌlnˈɛɹɪs"], "urethral": ["urethral", "jʊɹˈiθɹəl"], "uvulae": ["you view lee", "jˈuvjəli"], "vastus": ["vast us", "vˈæstəs"], "veli": ["vee lie", "vˈilI"], "vocalis": ["voe cal iss", "vOkˈælɪs"], "zygomaticus": ["zy go mat ick us", "zˌIɡOmˈæɾɪkəs"]}, "nickw": {"gastroc": ["gas trock", "ɡˈæstɹɑk"], "subscap": ["sub scap", "sˈʌbskæp"], "scap": ["scap", "skˈæp"], "rec": ["wreck", "ɹˈɛk"], "fem": ["fem", "fˈɛm"], "tib": ["tib", "tˈɪb"], "ant": ["ant", "ˈænt"], "glute": ["glute", "ɡlˈut"], "delts": ["delts", "dˈɛlts"], "lats": ["lats", "lˈæts"], "pecs": ["pecs", "pˈɛks"], "supra": ["soo pruh", "sˈupɹə"], "infra": ["in fruh", "ˈɪnfɹə"], "pec": ["peck", "pˈɛk"], "vmo": ["V M O", ""], "tfl": ["T F L", ""], "scm": ["S C M", ""], "ql": ["Q L", ""], "ta": ["T A", ""], "tva": ["T V A", ""]}, "gsay": ["the face", "the neck", "the shoulder", "the chest", "the back", "the abdomen", "the upper arm", "the forearm", "the hand", "the pelvis", "the hips and glutes", "the thigh", "the lower leg", "the foot", "the breathing muscles", "the muscles of the eye", "the ear", "the mouth, tongue and palate", "the throat", "the pelvic floor"], "phrases": {"intro": [{"t": "Find a comfortable position, and let your body be still. In this session, we'll move slowly through {g}.", "b": false}, {"t": "Welcome. Let your eyes close, or soften your gaze, and let's turn our attention to {g}.", "b": false}, {"t": "Settle in, and let the day fall away for a moment. Together, we'll explore {g}, one muscle at a time.", "b": false}, {"t": "Take a moment to arrive. Feel the weight of your body, fully supported. When you're ready, we'll begin with {g}.", "b": false}, {"t": "Let's begin. There's nothing to do except breathe and notice. Our focus today is {g}.", "b": false}], "settle": [{"t": "Take a slow breath in through your nose... and let it go.", "b": true}, {"t": "Breathe in deeply... and as you breathe out, let your shoulders drop.", "b": true}, {"t": "Let your breathing find its own easy rhythm.", "b": true}, {"t": "Breathe in for a count of four... and out, slowly, for a count of six.", "b": true}], "first": [{"t": "We'll start here, with the first muscle.", "b": false}, {"t": "Let your attention come to rest on this first muscle.", "b": false}, {"t": "Begin by bringing your awareness here.", "b": false}], "next": [{"t": "Breathe in, and on the out breath, we will recenter our minds on the next muscle.", "b": true}, {"t": "Now, gently move your attention to the next muscle.", "b": false}, {"t": "As you exhale, let your awareness travel onward.", "b": true}, {"t": "Next, we come to another muscle.", "b": false}, {"t": "Let that area soften, and turn your attention onward.", "b": false}, {"t": "Breathing in, and breathing out, we arrive at the next muscle.", "b": true}, {"t": "Without hurrying, shift your focus.", "b": false}, {"t": "Now let your awareness settle somewhere new.", "b": false}, {"t": "Take one easy breath, and move on.", "b": true}, {"t": "From here, we drift onward.", "b": false}], "hold": [{"t": "Notice any tension here, and let it soften.", "b": false}, {"t": "Simply notice how it feels, without needing to change anything.", "b": false}, {"t": "Imagine it growing warm and heavy.", "b": false}, {"t": "Let it rest.", "b": false}, {"t": "Picture where it begins and where it ends, and let the space between relax.", "b": false}, {"t": "Breathe into this area, and let it release.", "b": true}, {"t": "Let it be loose and easy.", "b": false}, {"t": "If you can, feel it gently lengthen as you breathe out.", "b": true}], "breath": [{"t": "Breathe in... and breathe out.", "b": true}, {"t": "Stay with the breath.", "b": true}, {"t": "If your mind wanders, that's okay. Gently come back.", "b": false}, {"t": "Soft, slow breaths.", "b": true}], "close": [{"t": "Now let your awareness widen, to hold all of {g} at once.", "b": false}, {"t": "Feel the whole of {g}, calm and at ease.", "b": false}, {"t": "That completes our journey through {g}.", "b": false}], "exit1": [{"t": "Take one more slow, full breath... and let it go.", "b": true}], "exit2": [{"t": "When you're ready, let your attention return to the room around you.", "b": false}, {"t": "Gently wiggle your fingers and toes, and open your eyes when you're ready.", "b": false}], "exit3": [{"t": "Carry this sense of calm with you.", "b": false}]}}''')
json.dump(DATA, open('/content/data.json', 'w', encoding='utf-8'), ensure_ascii=False)
print(len(DATA['muscles']), 'muscles')

In [ ]:
# 4. The builder
open('/content/audio_build.py', 'w').write('# ---- Muscle Trainer audio library builder (runs in the Colab notebook; tested in the sandbox with fake tones) ----\nimport json, os, re, subprocess, wave\nimport numpy as np\n\nRATE = 24000\nGAP = 0.45  # seconds of silence between clips inside a sprite file\n\n\ndef kokoro_words(text, words, extra):\n    """Wrap Latin words in Kokoro\'s [word](/phonemes/) override so they are said exactly as written in the table."""\n    def rep(m):\n        w = m.group(0); k = w.lower()\n        p = words.get(k, (None, None))[1] or extra.get(k, (None, None))[1]\n        return f"[{w}](/{p}/)" if p else w\n    return re.sub(r"[A-Za-z\']+", rep, text)\n\n\ndef spoken_name(name):\n    m = re.match(r"(.*) \\((.*)\\)$", name)\n    if not m: return name + "."\n    base, inner = m.groups()\n    return f"{base} of the foot." if inner == "foot" else f"{base}, also called {inner}."\n\n\ndef nick_plain(n):\n    return re.sub(r"\\s*\\(.*\\)", "", n)\n\n\ndef all_lines(data):\n    """Every line the app can ask for: key -> (sprite file, text for Kokoro)."""\n    W, X = data["words"], data["nickw"]\n    out = {}\n    for m in data["muscles"]:\n        f = f"g{m[\'group\']}.mp3"\n        out["name:" + m["id"]] = (f, kokoro_words(spoken_name(m["name"]), W, X))\n        out["act:" + m["id"]] = (f, m["action"] + ".")\n        if m.get("nick"):\n            out["nick:" + m["id"]] = (f, kokoro_words(f"Some people call it the {nick_plain(m[\'nick\'])}.", W, X))\n    for g, gs in enumerate(data["gsay"]):\n        for role in ("intro", "close"):\n            for p in data["phrases"][role]:\n                t = p["t"].replace("{g}", gs)\n                out["p:" + t] = (f"g{g}.mp3", t)\n    for role, ps in data["phrases"].items():\n        if role in ("intro", "close"): continue\n        for p in ps:\n            out["p:" + p["t"]] = ("phrases.mp3", p["t"])\n    return out\n\n\ndef trim(a, thr=0.01, pad=0.04):\n    idx = np.where(np.abs(a) > thr)[0]\n    if not len(idx): return a\n    s, e = max(0, idx[0] - int(pad * RATE)), min(len(a), idx[-1] + int(pad * RATE))\n    return a[s:e]\n\n\ndef build(data, tts, outdir="audio", bitrate="48k", log=print):\n    os.makedirs(outdir, exist_ok=True)\n    lines = all_lines(data)\n    files = {}\n    for k, (f, t) in lines.items(): files.setdefault(f, []).append((k, t))\n    clips = {}\n    gap = np.zeros(int(GAP * RATE), dtype=np.float32)\n    for n, (f, items) in enumerate(sorted(files.items())):\n        parts, pos = [gap], GAP\n        for k, t in items:\n            a = trim(np.asarray(tts(t), dtype=np.float32))\n            clips[k] = [f, round(pos, 3), round(len(a) / RATE, 3)]\n            parts += [a, gap]; pos += len(a) / RATE + GAP\n        audio = np.concatenate(parts)\n        wav = os.path.join(outdir, f[:-4] + ".wav")\n        with wave.open(wav, "wb") as w:\n            w.setnchannels(1); w.setsampwidth(2); w.setframerate(RATE)\n            w.writeframes((np.clip(audio, -1, 1) * 32767).astype("<i2").tobytes())\n        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-ac", "1", "-b:a", bitrate, os.path.join(outdir, f)], check=True)\n        os.remove(wav)\n        log(f"{n + 1}/{len(files)}  {f}: {len(items)} clips, {pos / 60:.1f} min")\n    man = {"version": 1, "voice": data.get("voice", ""), "clips": clips}\n    json.dump(man, open(os.path.join(outdir, "manifest.json"), "w"), ensure_ascii=False)\n    log(f"done: {len(clips)} clips in {len(files)} files")\n    return man\n')
print('builder ready')

In [ ]:
# 5. Make the library (the long step)
!rm -rf /content/audio /content/audio.zip
!/content/kenv/bin/python -u /content/kgen.py {VOICE} {SPEED} build
!cd /content && zip -qr audio.zip audio && ls -la audio | head -30
from google.colab import files
files.download('/content/audio.zip')

**If a word sounds wrong:** tell Claude which one; the fix is one line in the table, then run again.